In [ ]:
import sys
sys.path.append("..")
sys.path.append("../pipeline")

import os
import time
import warnings

warnings.filterwarnings("ignore", message="PySpark does not yet fully support pandas", category=FutureWarning)

from constants import QUERIES_DIR, get_scenario_paths
from pipeline.spark_session import (
    create_spark_session,
    register_tables_in_spark,
    apply_run_options
)
from pipeline.config_loader import options
from pipeline.benchmark import _print_run_configuration

In [ ]:
spark = create_spark_session("Benja-Test")
run_configuration = apply_run_options(spark, options)
_print_run_configuration(run_configuration)

In [ ]:
# conf_df = spark.sql("SET -v").toPandas()   # columns: key, value, meaning, Since version
# conf_df.to_excel("spark_conf.xlsx", index=False)

In [ ]:
SCALE_FACTOR = 1
_, parquet_dir, _ = get_scenario_paths(SCALE_FACTOR)

In [ ]:
register_tables_in_spark(
    spark,
    parquet_dir,
    options.spark.register_tables_as,
    analyze=options.spark.cbo_enabled,
)

In [ ]:
QUERY_NUMBER = 72

QUERY_FILE_ORIGINAL = os.path.join(
    QUERIES_DIR,
    "original",
    f"queries_{SCALE_FACTOR}GB",
    f"query{QUERY_NUMBER}.sql"
)
QUERY_FILE_CHANGED = os.path.join(
    QUERIES_DIR,
    "changed",
    f"queries_{SCALE_FACTOR}GB",
    f"query{QUERY_NUMBER}.sql"
)

N_RUNS = 3

In [ ]:
QUERY_SQL_ORIGINAL = open(QUERY_FILE_ORIGINAL, "r").read()

for run in range(N_RUNS):
    print(f"Run {run + 1} of {N_RUNS}")
    start_time = time.time()
    result_original = spark.sql(QUERY_SQL_ORIGINAL).toPandas()
    end_time = time.time()
    print(f"Time taken for original query: {end_time - start_time:.2f} seconds")

spark._jvm.System.gc()

In [ ]:
def print_join_order(sql):
    plan = spark.sql(sql)._jdf.queryExecution().optimizedPlan()

    def children(node):
        cs = node.children()
        return [cs.apply(i) for i in range(cs.size())]

    def tables(node):
        # Identify the tables under a node by the column prefix of each leaf (cs, inv, w, i, ...)
        kids = children(node)
        if not kids:
            return {node.output().apply(0).name().split("_")[0]}
        return set().union(*(tables(k) for k in kids))

    joins = []
    def visit(node):
        for k in children(node):  # children first, so joins come out in execution order
            visit(k)
        if node.nodeName() == "Join":
            joins.append(node)

    visit(plan)
    for n, j in enumerate(joins, 1):
        left, right = children(j)
        cond = j.condition().get().sql() if j.condition().isDefined() else "(no condition)"
        print(f"{n}. {j.joinType()}: {sorted(tables(left))} ⋈ {sorted(tables(right))}")
        print(f"   ON {cond}")
        print()

print_join_order(QUERY_SQL_ORIGINAL)

In [ ]:
QUERY_SQL_CHANGED = open(QUERY_FILE_CHANGED, "r").read()

for run in range(N_RUNS):
    print(f"Run {run + 1} of {N_RUNS}")
    start_time = time.time()
    result_changed = spark.sql(QUERY_SQL_CHANGED).toPandas()
    end_time = time.time()
    print(f"Time taken for changed query: {end_time - start_time:.2f} seconds")

spark._jvm.System.gc()


In [ ]:
print_join_order(QUERY_SQL_CHANGED)

In [ ]:
result_original.equals(result_changed)